# FRCRN_SE

In [1]:
import warnings
from pathlib import Path
from tqdm import tqdm
import os
import gc
import time
import numpy as np
import soundfile as sf
from scipy import signal
import torch
from clear_memory import clear_memory

warnings.filterwarnings('ignore')

In [2]:
input_dir = Path('../ad_detection/data/raw/Pitt')
output_dir = Path('../ad_detection/data/denoised/Pitt-FRCRN_SE')

control_files = list((input_dir / 'Control').glob('*.wav'))
dementia_files = list((input_dir / 'Dementia').glob('*.wav'))

## Load Model

In [3]:
from clearvoice import ClearVoice

model_name = 'FRCRN_SE_16K'
target_sr = 16000  # 目标采样率，必须与模型匹配

myClearVoice = ClearVoice(
    task='speech_enhancement',
    model_names=[model_name]
)

## Denoise Function

In [4]:
def denoise_audio(audio_path, model, target_sr=16000):
    """
    使用 ClearerVoice 进行语音降噪和增强
    
    Args:
        audio_path: 输入音频文件路径
        model: ClearVoice 模型实例
        target_sr: 目标采样率（16000 或 48000，取决于模型）
    
    Returns:
        denoised_audio: 降噪后的音频 numpy array
        sr: 采样率
    """
    # 加载音频
    audio, sr = sf.read(str(audio_path))
    
    # 如果是多声道，先转为单声道（在重采样之前）
    if len(audio.shape) == 2:
        # audio 形状是 (samples, channels)
        audio = np.mean(audio, axis=1)
    
    # 重采样到目标采样率（使用 scipy，更稳定）
    if sr != target_sr:
        # 计算目标样本数
        num_samples = int(len(audio) * target_sr / sr)
        audio = signal.resample(audio, num_samples)
    
    # 确保是 float32 类型
    audio = audio.astype(np.float32)
    
    # 转换为 [batch, length] 格式
    audio = np.reshape(audio, [1, audio.shape[0]])
    
    # 应用 ClearVoice 降噪
    # 使用 torch.no_grad() 禁用梯度计算，节省显存
    with torch.no_grad():
        # online_write=False 表示返回 numpy 数组而不是直接写入文件
        output_wav = model(audio, online_write=False)
    
    # output_wav 形状: [batch, length]
    return output_wav[0, :], target_sr

In [5]:
def batch_denoise(files, output_subdir, model, target_sr, group_name):
    """
    批量降噪处理（每个文件前后都清理显存）
    
    Args:
        files: 待处理的音频文件列表
        output_subdir: 输出子目录
        model: ClearVoice 模型实例
        target_sr: 目标采样率
        group_name: 组名（用于显示进度）
    """
    # 创建输出目录
    output_subdir.mkdir(parents=True, exist_ok=True)
    
    success_count = 0
    skip_count = 0
    fail_count = 0
    
    for audio_file in tqdm(files, desc=f"Processing {group_name}"):
        output_file = output_subdir / audio_file.name
        
        # 跳过已处理的文件
        if output_file.exists():
            skip_count += 1
            continue
        
        try:
            # ⚡ 处理前清理显存
            clear_memory()
            
            # 降噪
            denoised_audio, sr = denoise_audio(audio_file, model, target_sr)
            
            # 保存（16位整数格式）
            sf.write(str(output_file), denoised_audio, sr, subtype='PCM_16')
            success_count += 1
            
            # ⚡ 处理后立即清理显存
            del denoised_audio  # 删除大数组
            clear_memory()
            
        except Exception as e:
            fail_count += 1
            print(f"\nFailed: {audio_file.name}: {e}")
            # ⚡ 失败后也要清理显存
            clear_memory()
    
    # 打印统计信息
    print(f"\n{group_name} 处理完成:")
    print(f"成功: {success_count}")
    print(f"跳过: {skip_count}")
    print(f"失败: {fail_count}")
    print(f"总计: {len(files)}")

## Execute denoise function

In [6]:
clear_memory()

batch_denoise(
    dementia_files,
    output_dir / 'Dementia',
    myClearVoice,
    target_sr=target_sr,
    group_name='Dementia'
)

clear_memory()

batch_denoise(
    control_files,
    output_dir / 'Control',
    myClearVoice,
    target_sr=target_sr,
    group_name='Control'
)

Processing Dementia: 100%|██████████| 309/309 [00:00<00:00, 139239.36it/s]



Dementia 处理完成:
成功: 0
跳过: 309
失败: 0
总计: 309


Processing Control: 100%|██████████| 242/242 [00:00<00:00, 124756.83it/s]


Control 处理完成:
成功: 0
跳过: 242
失败: 0
总计: 242
